# Assignment 4: Recurrent Neural Networks

## Part 1: LSTM

### Step 0: Import Libraries

In [1]:
import torch
from datasets import load_dataset
from collections import Counter
import re
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

In [2]:
import warnings
warnings.filterwarnings(action='ignore')

### Step 1: Data Loading and Preprocessing (12 marks)

For this assignment, we will be using the imdb dataset from the 🤗 Datasets library

In [3]:
# TO DO: Load the dataset (1 mark)
dataset = load_dataset("imdb")

sample_dataset = [dataset["train"][i] for i in range(3)]  # Sample dataset to test functions 
print(sample_dataset)

[{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far b

We need to preprocess the data before we can feed it into the model. The first step is to define a custom tokenizer to perform the following tasks: 
- Extract the text data from the dataset
- Remove any non-alphanumeric characters
- Separate each data sample into separate words (tokens)

In [4]:
def tokenizer(data_iter):
    '''Tokenizes the input data
    input: data_iter (type: dictionary)
    output: text (type: list[list[str]])
    '''
    # TO DO: fill in this function (2 marks)
    tokens = []
    for data in data_iter:   # Iterate through dictionary
        text = data["text"]  # Get text data from dictionary

        text = re.sub(r'<.*?>', ' ', text)          # Remove HTML tags 
        text = re.sub(r'[^A-Za-z0-9]+', " ", text)  # Keep alnum chars
        tokens.append(text.split())               # Split into word tokens

    return tokens

print(tokenizer(sample_dataset))

[['I', 'rented', 'I', 'AM', 'CURIOUS', 'YELLOW', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was', 'first', 'released', 'in', '1967', 'I', 'also', 'heard', 'that', 'at', 'first', 'it', 'was', 'seized', 'by', 'U', 'S', 'customs', 'if', 'it', 'ever', 'tried', 'to', 'enter', 'this', 'country', 'therefore', 'being', 'a', 'fan', 'of', 'films', 'considered', 'controversial', 'I', 'really', 'had', 'to', 'see', 'this', 'for', 'myself', 'The', 'plot', 'is', 'centered', 'around', 'a', 'young', 'Swedish', 'drama', 'student', 'named', 'Lena', 'who', 'wants', 'to', 'learn', 'everything', 'she', 'can', 'about', 'life', 'In', 'particular', 'she', 'wants', 'to', 'focus', 'her', 'attentions', 'to', 'making', 'some', 'sort', 'of', 'documentary', 'on', 'what', 'the', 'average', 'Swede', 'thought', 'about', 'certain', 'political', 'issues', 'such', 'as', 'the', 'Vietnam', 'War', 'and', 'race', 'issues', 'in', 'the', 'United', 'St

We will also need to extract the labels from the dataset. Complete the label_extractor function below:

In [5]:
def label_extractor(data_iter):
    '''Takes the label for each data sample and stores it in a separate list
    input: data_iter (type: dictionary)
    output: labels (type: list)
    '''
    # TO DO: fill in this function (1 mark)
    labels = []
    for data in data_iter:
        label = data["label"]  # Get label data from dictionary
        labels.append(label)   # Add to empty list
    return labels

print(label_extractor(sample_dataset))

[0, 0, 0]


Now that we have the text data separated into words, we need to define the vocabulary. We cannot keep all the words in the vocabulary, so we want to limit the vocabulary size and only take the most common words. In this case, the maximum vocabulary size is 10,000 words. Any word that is excluded will be set to an unknown token. You can use the function below to build the vocabulary:

In [6]:
# Build a vocabulary
def build_vocab(data_iter, max_size=10000):
    '''Creates a vocabulary based on the training data
    input: data_iter (type: list[list[str]])
    output: vocab (type: dictionary)
    '''
    counter = Counter()
    for words in data_iter:
        counter.update(words)
    # Filter to most common words
    vocab = {word: i + 1 for i, (word, _) in enumerate(counter.most_common(max_size))}
    # Add a token for unknown words (0)
    vocab['<unk>'] = 0 
    return vocab

In the vocabulary, each word is mapped to a number in the vocabulary. We will need to encode the dataset based on these numbers, as tensors cannot handle string data.

The next step is to pad or truncate each sequence based on a maximum length, to make sure that the dataset can be transformed into a tensor (as discussed in class).

Fill in the function below to encode and pad the dataset:

In [7]:
def encode_and_pad(text, vocab, max_len=100):
    '''Encode and pad the input text dataset
    input: text (type: list[list[str]])
    input: vocab (type: dictionary)
    input: max_len (type: int)
    output: texts (type: list[list[str]])
    '''
    # TO DO: fill in the function to encode text to integers and pad/truncate sequences (2 marks)
    encoded_texts = []
    for words in text:      # Loop through list of text
        encoded = []
        for word in words:  # Loop through list of words
            encoded.append(vocab.get(word, vocab['<unk>']))  # Add words index from vocab (use unk if missing)

        encoded = encoded[:max_len]  # Truncate if max length is exceeded

        padding = max_len - len(encoded)  # Amount of padding needed
        if padding > 0:
            encoded += [0] * padding  # Pad with 0

        encoded_texts.append(encoded)

    return encoded_texts

The next step is to create a custom PyTorch Dataset class that calls the `encode_and_pad()` function and stores the text and labels as tensors. Fill in the `init` portion of the class: 

In [8]:
# Create a custom PyTorch Dataset class
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len):
        # TO DO: call the encode_and_pad() function and set self.texts and self.labels (2 marks)
        encoded = encode_and_pad(texts, vocab, max_len)        # Encode text data based on vocab and pad if needed
        
        self.texts = torch.tensor(encoded, dtype=torch.long)   # Convert encoded text data to tensor
        self.labels = torch.tensor(labels, dtype=torch.float)  # Convert labels to tensors

    def __len__(self): 
        return len(self.labels)
    def __getitem__(self, idx): 
        return self.texts[idx], self.labels[idx]

Now you can call all the functions that have been created:

In [9]:
MAX_LEN = 256 # Sequence length
BATCH_SIZE = 64

# TO DO: Tokenize training data (1 mark)
train_tokens = tokenizer(dataset["train"])
test_tokens  = tokenizer(dataset["test"])

# TO DO: Extract labels from training and testing data (1 mark)
train_labels = label_extractor(dataset["train"])
test_labels  = label_extractor(dataset["test"])

# TO DO: Build Vocabulary (from training data only) (1 mark)
vocab = build_vocab(train_tokens, max_size=10000)

# TO DO: Prepare datasets (using TextDataset class) and store datasets using DataLoaders (1 mark)
train_dataset = TextDataset(train_tokens, train_labels, vocab, MAX_LEN)
test_dataset  = TextDataset(test_tokens,  test_labels,  vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

print(train_dataset.texts[0])
print(train_dataset.labels[0])

tensor([   8, 1662,    8,    0,    0,    0,   37,   72,  400, 1138,   85,    4,
          33,    1, 7650,   10, 3544,    9,   59,    9,   13,   89,  648,    7,
        7229,    8,   94,  575,   10,   32,   89,    9,   13,    0,   31, 1578,
         761,    0,   62,    9,  131,  807,    5, 3655,   12,  731, 2134,  114,
           3,  346,    4,  107, 1205, 3212,    8,   64,   65,    5,   66,   12,
          18,  562,   14,  112,    6, 4324,  188,    3,  209, 4389,  474, 1517,
         794, 4722,   35,  503,    5,  866,  337,   61,   50,   43,  116,  126,
         850,   61,  503,    5, 1149,   41,    0,    5,  240,   49,  450,    4,
         677,   22,   54,    1,  882,    0,  196,   43,  842,  994, 1359,  145,
          15,    1, 2661,  951,    2, 1641, 1359,    7,    1, 2627, 2580,  126,
         197, 2288, 8262,    2, 2072,    0,    4,    0,   43,   67, 5037,   22,
        2566,   61,   45,  403,   17,   41,  474, 1767, 8802,    2, 1059,  388,
         202, 1140,   71,   43,    8,   

### Step 2: Define Model (4 marks)

For this assignment, we will be using the LSTM model. Inside the LSTM model, the first layer will be an embedding layer, to convert the singular numerical representation of each word into an embedded vector. We can use `nn.Embedding(...)` for this.

Define LSTMClassifier below:

In [10]:
# TO DO: Define LSTM class (4 marks)
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers=1):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # TO DO: Embedding layer 
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # TO DO: LSTM layer
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True)
         
        # TO DO: Linear fully-connected layer
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # TO DO: Fill in the model steps
        # NOTE: The LSTM outputs (output, (hidden, cell)) - hidden and cell are not used
        # NOTE: Use the hidden state from the final time step for the fc layer
        
        x = self.embedding(x)
        out, (hn, cn) = self.lstm(x)
        h = hn[-1]
        out = self.fc(h)
        return out

### Step 3: Define Training and Testing Loops (4 marks)

The next step is to define functions for the training and testing loops. For this case, we will only be calculating the loss at each epoch.

In [11]:
# TO DO: Define training loop (2 marks)
def train_loop(dataloader, model, loss_fn, optimizer, device):
    model.train()
    train_loss = 0.0
    correct = 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(device).unsqueeze(1)  # Move the data to the device being used
        output = model(X)                  # Make a prediction
        loss = loss_fn(output, y)          # Calculate Loss

        optimizer.zero_grad()  # Reset Gradients
        loss.backward()        # Determine new Gradients
        optimizer.step()       # Update Weights

        train_loss += loss.item() * y.size(0)  # Batch Loss
        pred = (torch.sigmoid(output) >= 0.5)  # Binary prediction
        correct += pred.eq(y).sum().item()     # Count the number of correct predictions

    avg_loss = train_loss / len(dataloader.dataset)   # Average loss over dataset

    return avg_loss

In [12]:
# TO DO: Define testing loop (2 marks)
def test_loop(dataloader, model, loss_fn, device):
    model.eval()  # Evaluate mode
    test_loss = 0.0
    correct = 0

    with torch.no_grad():  # No gradients computed
        for X, y in dataloader:
            X, y = X.to(device), y.to(device).unsqueeze(1)
            output = model(X)

            test_loss += loss_fn(output, y).item() * y.size(0)
            pred = (torch.sigmoid(output) >= 0.5)
            correct += pred.eq(y).sum().item()

    # Determine the loss and accuracy over the dataset
    test_loss /= len(dataloader.dataset)

    return test_loss

### Step 4: Train and Evaluate (3 marks)

Now that we have all the necessary functions, we can select our hyperparameters, and train and evaluate our model. For this case, since we are not comparing different models, we do not need a validation set.

In [13]:
# Hyperparameters
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
OUTPUT_DIM = 1 # Binary classification
NUM_LAYERS = 1

In [14]:
# TO DO: Create model object (1 mark)
model = LSTMClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, NUM_LAYERS)

In [15]:
import torch.optim as optim

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

LSTMClassifier(
  (embedding): Embedding(10001, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

Since this case is binary optimization, we will use the binary cross entropy criterion, `BCEWithLogitsLoss()`. This model is similar to Cross Entropy, but uses a sigmoid layer instead of a softmax layer. For the optimization function, we will use Adam with a learning rate of 0.01.

In [16]:
# TO DO: Define optimization model and criterion (1 mark)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()

We can now run our training and testing loops. Since this takes a long time to run, we will set the number of epochs to 5. Print out the training and testing losses.

In [17]:
# TO DO: Run training and testing loops and print losses for each epoch (1 mark)
epochs = 5

# Lists to store loss values
train_losses, test_losses = [], []

# Loop over a set number of epochs
print("Epoch  Train Loss  Testing Loss")
for epoch in range(epochs):
    # Call Train and Validation loops
    train_loss = train_loop(train_loader, model, criterion, optimizer, device)
    test_loss = test_loop(test_loader, model, criterion, device)

    # Add current epochs loss to lists
    train_losses.append(train_loss)
    test_losses.append(test_loss)

    print(f"{epoch+1:<7}{train_loss:<12.4f}{test_loss:<17.4f}")

Epoch  Train Loss  Testing Loss
1      0.6953      0.6945           
2      0.5940      0.5961           
3      0.4742      0.5161           
4      0.3878      0.5040           
5      0.3406      0.4891           


## Part 2: Questions and Process Description

### Questions (12 marks)

1. Do you think this model worked well to classify the data? Why or why not? Can you make a good decision about this only using loss data?
    - From the training and testing loss, it can be seen that the model begins to classify well on the training data over the epochs, but that the testing loss stagnates on new unseen data. This suggests that the model is beginning to overfit, as it struggles to generalize and incorrectly classifies reviews it has not seen before. Overall, it is difficult to determine whether the model works well using only loss, and more metrics such as accuracy would be needed to conclude the models performance. However, based on the testing loss slowing down around 0.50-0.49 while training loss continues to improve, we can say that the model is starting to overfit and will need to be improved.

1. What could you do to further improve the results? Provide two suggestions.
    - To improve results, one thing that can be done to improve generalization and reduce the overfitting observed, is to add dropout at the fully connected layer. This ensures that the model does not rely too heavily on specific pathways when making a prediction, which in turn encourages better generalization. Another improvement would be to increase the number of LSTM layers in the model. Adding an additional layer allows the model to learn more complex patterns in the text, which can improve its ability to classify reviews more accurately.

1. Why does a simple RNN often underperform compared to LSTM or GRU on long text sequences such as IMDB reviews?
    - On long text sequences, RNNs tend to underperform compared to LSTM and GRU models as they can suffer from vanishing gradients, which makes it difficult for the network to retain important information from earlier timesteps during training. Since LSTM and GRU models use gating mechanisms to control information flow, they are able to handle longer dependencies and therefore longer text data such as IMDB reviews more effectively.

1. Why does the embedding layer improve performance compared to one-hot encoding?
    - One-hot encoding represents each word as a vector where all elements are 0 except for a single 1 corresponding to that word, which can make the vector incredibly long. This means there is no inherent relationship between words with most of the vector being unused, making learning slow and inefficient. Using an embedding layer on the other hand, represents each word as a smaller continuous vector where the values are learned so that similar words have similar representations. This allows the model to find relationships and patterns between words with similar meanings more effectively, leading to better performance and faster learning compared to one-hot encoding.

1. If we switched to character-level input instead of word-level, what changes would we expect in performance and training time?
    - Switching to character-level instead of word-level inputs would cause the model to learn patterns at a much finer level, which would also increase the sequence length. As a result, training time would be much longer and the model would take more epochs to learn meaningful relationships. While character-level models can capture details like spelling and punctuation, they often perform worse overall because they have a harder time understanding the full meaning of words and sentences. Therefore, by switching to character-level inputs, we can expect worse overall performance along with longer training times.

1. How does vocabulary size influence model performance and generalization?
    - When training a model for text classification, large or small vocabulary sizes can affect both performance and generalization negatively. A very large vocabulary can make the model too complex, which may improve performance on training data but cause overfitting and worse generalization to new data. A smaller vocabulary may miss important relationships between words or concepts, reducing overall performance. Therefore, it is important to choose the right vocabulary size to help the model learn meaningful patterns while generalizing well to unseen data.

### Process Description (4 marks)
Please describe the process you used to create your code. Cite any websites or generative AI tools used. You can use the following questions as guidance:
1. Where did you source your code?
    - The code for this assignment was sourced from class notes, examples posted on D2L, and previous assignments I have completed. More specifically, the char_RNN_classification_tutorial and char_RNN_generation_tutorial helped me understand the workflows of RNNs, which made it easier to see how the LSTM model should be created.

1. In what order did you complete the steps?
    - The steps for this assignment were completed in order, beginning with preprocessing the data, followed by creating the model, and finally training/testing, and evaluating it. Completing the assignment this way helped me understand the overall process of building an LSTM model and the steps needed to apply it to text data for binary classification.

1. If you used generative AI, what prompts did you use? Did you need to modify the code at all? Why or why not?
    - When working through this assignment, I used generative AI mainly to help me understand certain concepts related to RNNs. For example, by asking questions such as “What is BCEWithLogitsLoss() and how does it function?” and “Why is padding needed when working with text data?” helped me better understand preprocessing and how to apply the LSTM model to the dataset.

1. Did you have any challenges? If yes, what were they? If not, what helped you to be successful?
    - Overall, I did not run into many challenges while completing this assignment. By consistently referring to the notes, examples provided, and previous assignments, I was able to implement each part successfully and gain a deeper understanding of how LSTM models function and how they can be applied.

## Part 3: Reflection (2 marks)
Include a sentence or two about:
- what you liked or disliked,
- found interesting, confusing, challenging, motivating while working on this assignment.
    - This was a great assignment in helping me understand RNNs and LSTM models on a deeper level. I especially enjoyed working with the text data and learning how to preprocess it for training and testing, allowing me to see how it can be applied in real world scenarios.